# U - Understanding the Data (Datenverständnis)

## QUA³CK-Phase

Die U-Phase untersucht Struktur, Qualität, Verteilungen, Anomalien und
Zusammenhänge der Daten. Die Erkenntnisse begründen Datenbereinigung,
Merkmalsbildung und Algorithmuswahl.

## Umsetzung im Projekt

Das Projekt verbindet stündliche PV-Erzeugung und installierte PV-Leistung von
SMARD mit Wetterbeobachtungen räumlich verteilter DWD-Stationen. Der finale
Datensatz liegt auf Deutschlandebene in UTC vor.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

pd.set_option("display.max_columns", 30)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from pv_weather import TARGET, add_features, load_project_data
from pv_weather.modeling import select_pv_relevant_hours

sns.set_theme(style="whitegrid")

data, source = load_project_data(
    ROOT / "data" / "processed" / "hourly_pv_weather.csv"
)
featured = add_features(data)
daylight = select_pv_relevant_hours(featured)

print(source)
print(f"Zeitraum: {data['timestamp_utc'].min()} bis {data['timestamp_utc'].max()}")
print(f"Vollständiges Panel: {len(data):,} Stunden")
print(f"PV-relevante Tageslichtstunden: {len(daylight):,}")
display(data.head())


## Datenherkunft und Datenfluss

| Daten | Quelle | Projektumsetzung |
|---|---|---|
| PV-Erzeugung | Bundesnetzagentur / SMARD | Download und Einlesen in `download.py` / `ingest.py` |
| Installierte PV-Leistung | Bundesnetzagentur / SMARD | jährliche Zuordnung zur Normierung |
| Strahlung, Temperatur, Feuchte, Wolken, Wind | Deutscher Wetterdienst | Stationsauswahl, Stundenaggregation und Mittelung |
| Gemeinsames Stundenpanel | projektintern | `data/processed/hourly_pv_weather.csv` |

Der automatische End-to-End-Ablauf ist in `pv_weather/workflow.py` gekapselt.
Fehlen lokale Realdaten, stellt `load_project_data` einen ausdrücklich als
synthetisch markierten Demodatensatz bereit.


## Schema- und Qualitätsprüfung


In [ ]:
quality = pd.DataFrame(
    {
        "Datentyp": data.dtypes.astype(str),
        "Fehlend (n)": data.isna().sum(),
        "Fehlend (%)": data.isna().mean().mul(100).round(2),
        "Eindeutige Werte": data.nunique(),
    }
)
display(quality)

checks = pd.Series(
    {
        "Zeitstempel monoton": data["timestamp_utc"].is_monotonic_increasing,
        "Doppelte UTC-Stunden": int(data["timestamp_utc"].duplicated().sum()),
        "Ungültige Zielwerte": int(
            (~featured[TARGET].between(0, 1.2) & featured[TARGET].notna()).sum()
        ),
        "Negative Globalstrahlung": int((data["global_radiation_j_cm2"] < 0).sum()),
        "Feuchte außerhalb 0-100 %": int(
            (~data["relative_humidity_pct"].between(0, 100)
             & data["relative_humidity_pct"].notna()).sum()
        ),
    },
    name="Prüfergebnis",
)
display(checks.to_frame())


Nullwerte sind nicht automatisch fehlend: Globalstrahlung, Diffusstrahlung,
Sonnenscheindauer und PV-Erzeugung dürfen nachts physikalisch korrekt null sein.
Für die Modellierung werden nur Stunden mit Globalstrahlung über 10 J/cm² und
einem Sonnenzenit unter 90° verwendet; das vollständige Panel bleibt für
Datenkontrolle und Exploration erhalten.


In [ ]:
numeric_summary = featured[
    [
        "temperature_c",
        "relative_humidity_pct",
        "global_radiation_j_cm2",
        "cloud_cover_oktas",
        "wind_speed_m_s",
        "estimated_module_temperature_c",
        TARGET,
    ]
].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T
display(numeric_summary.round(3))


## Explorative Zusammenhänge


In [ ]:
plot_data = daylight.dropna(
    subset=["global_radiation_j_cm2", "temperature_c", TARGET]
).sample(min(8000, len(daylight)), random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
scatter = axes[0].scatter(
    plot_data["global_radiation_j_cm2"],
    plot_data["temperature_c"],
    c=plot_data[TARGET] * 100,
    cmap="YlOrRd",
    s=14,
    alpha=0.45,
)
fig.colorbar(scatter, ax=axes[0], label="Normierte PV-Erzeugung (%)")
axes[0].set(
    title="Einstrahlung, Temperatur und Erzeugung",
    xlabel="Globalstrahlung (J/cm²)",
    ylabel="Lufttemperatur (°C)",
)

corr_columns = [
    "temperature_c",
    "relative_humidity_pct",
    "global_radiation_j_cm2",
    "cloud_cover_oktas",
    "wind_speed_m_s",
    "estimated_module_temperature_c",
    "diffuse_share",
    TARGET,
]
sns.heatmap(
    daylight[corr_columns].corr(),
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    ax=axes[1],
)
axes[1].set_title("Lineare Korrelationen in Tageslichtstunden")
plt.tight_layout()
plt.show()


In [ ]:
high_radiation_limit = daylight["global_radiation_j_cm2"].quantile(0.75)
strong_sun = daylight[
    daylight["global_radiation_j_cm2"] >= high_radiation_limit
].copy()
strong_sun["Temperaturklasse"] = pd.cut(
    strong_sun["temperature_c"],
    [-np.inf, 15, 25, 30, np.inf],
    labels=["< 15 °C", "15-25 °C", "25-30 °C", ">= 30 °C"],
)
temperature_summary = (
    strong_sun.groupby("Temperaturklasse", observed=True)[TARGET]
    .agg(Stunden="count", Mittelwert="mean", Median="median")
)
temperature_summary[["Mittelwert", "Median"]] *= 100
display(temperature_summary.round(2))


## Interpretation und Grenzen der U-Phase

- Globalstrahlung ist fachlich und empirisch der dominante Treiber.
- Der Temperatureffekt darf nicht aus einer unkontrollierten Gesamt-Korrelation
  abgeleitet werden, weil starke Einstrahlung zugleich Module erwärmt und die
  Erzeugung erhöht.
- Temperaturklassen bei ähnlich starker Einstrahlung liefern nur deskriptive
  Hinweise, keinen Kausalnachweis.
- Das ungewichtete Stationsmittel kann Regionen mit vielen Stationen
  übergewichten und regionale PV-Leistungsunterschiede glätten.
- Jährliche Kapazitätswerte bilden den unterjährigen Ausbau nur näherungsweise
  ab.

## Technische Verankerung der U-Phase

- `pv_weather/download.py`: automatischer Bezug amtlicher Rohdaten
- `pv_weather/ingest.py`: Einlesen, Zeitvereinheitlichung und Aggregation
- `pv_weather/data.py`: Schema, Plausibilisierung und Daten-Fallback
- `pv_weather/features.py`: Zielvariable und abgeleitete Analysemerkmale
- `data/processed/hourly_pv_weather.csv`: kanonisches Stundenpanel

## Übergabe an A³

Die A³-Phase erhält eine kontinuierliche Zielvariable, einen zeitlich geordneten
Datensatz und die fachliche Anforderung, Strahlungs- und Temperatureffekte
gemeinsam abzubilden, ohne Kalendermerkmale als versteckte Stellvertreter zu
verwenden.
